# Philippine Traffic Sign Detection and Recognition Using YOLOv8n
## 1. Overview and objectives
Detect and recognize five Philippine traffic signs in static images and webcam frames; display bounding boxes, confidence, and rule-based suggestions.

In [ ]:
from pathlib import Path
import os,sys,json,hashlib
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'project_config.py').exists())
sys.path.insert(0,str(ROOT))
os.environ.setdefault('YOLO_CONFIG_DIR',str(ROOT/'outputs/ultralytics'))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'outputs/matplotlib'))
import numpy as np,pandas as pd,matplotlib.pyplot as plt,yaml,torch
from IPython.display import display,Image
from project_config import *
from decision_rules import SIGN_DECISIONS,NO_DECISION,decision_text,get_suggested_decision
from ultralytics import YOLO
print('Final checkpoint SHA-256:',verify_model())
model=YOLO(str(MODEL_PATH));assert list(model.names.values())==CLASSES
print('Classes:',CLASSES,'| confidence:',CONFIDENCE,'| NMS IoU:',NMS_IOU,'| imgsz:',IMAGE_SIZE)
evidence=json.loads((ROOT/'evaluation/selected_local_results.json').read_text())
def figure(relative,width=900):
    p=ROOT/relative
    if p.exists():display(Image(filename=str(p),width=width))
    else:print('Unavailable:',relative)


## 2. Dataset sources and class mapping
Dataset 1: original Roboflow source; relevant source IDs 22 Stop_Sign → 0 STOP, 8 Give Way → 1 Give Way, 14 No Entry → 2 No Entry, 26 Turn left ahead → 3 Turn Left, 27 Turn right ahead → 4 Turn Right. 
Dataset 2: from NSIT Traffic Sign Datased that is used for external testing only.

In [ ]:
display(pd.DataFrame({'Source ID':[22,8,14,26,27],'Target ID':range(5),'Target label':CLASSES}))
from notebook_support import audit_partition,image_files
audits={split:audit_partition(ROOT/f'dataset/images/{split}',ROOT/f'dataset/labels/{split}') for split in ['train','val','test']}
for split,a in audits.items():
    assert not a['issues'] and not a['multiclass'],(split,a['issues'])
display(pd.DataFrame({s:a['counts'] for s,a in audits.items()},index=CLASSES))


## 3. Original distribution and balancing
The original training pool contained 1,537 images: [121,425,332,333,326]. Seed42 balancing selected 121/class into 605 images. Validation remains 386 [32,107,83,84,80]. The unseen external test is 265 [53/class].

In [ ]:
distribution=pd.DataFrame({'Original training':[121,425,332,333,326],'Balanced training':audits['train']['counts'],'Validation':audits['val']['counts'],'External test':audits['test']['counts']},index=CLASSES)
display(distribution)
ax=distribution[['Balanced training','Validation','External test']].plot.bar(figsize=(10,4),rot=15)
ax.set_ylabel('Images');ax.set_title('Final dataset distribution');plt.tight_layout();plt.show()


## 4. Preparation and preprocessing
YOLO labels use class ID and normalized center-x, center-y, width and height. The packaged labels are remapped to 0–4. Input is letterboxed to640, normalized internally by Ultralytics. No horizontal/vertical flip during training preserves left/right semantics. 

In [ ]:
manifest=json.loads((ROOT/'dataset/manifest.json').read_text())
assert all(hashlib.sha256((ROOT/r['path']).read_bytes()).hexdigest()==r['sha256'] for r in manifest)
seen={};overlaps=[]
for r in manifest:
    if '/images/' not in r['path']:continue
    if r['sha256'] in seen and seen[r['sha256']]!=r['partition']:overlaps.append(r['path'])
    seen[r['sha256']]=r['partition']
print('Manifest files:',len(manifest),'| exact cross-split duplicates:',len(overlaps))


## 5. YOLOv8n architecture and transfer learning
YOLOv8n is a compact one-stage detector with a feature-extraction backbone, multiscale feature aggregation, and an anchor-free detection head. The user's run initialized from pretrained yolov8n.pt and fine-tuned for five classes.

In [ ]:
head=model.model.model[-1]
print('Parameters:',sum(p.numel() for p in model.model.parameters()),'| head:',type(head).__name__,'| classes:',head.nc,'| strides:',head.stride.tolist())
assert model.names==dict(enumerate(CLASSES))


## 6. Actual local training configuration and results


In [ ]:
args=yaml.safe_load((ROOT/'training/training_configuration.yaml').read_text())
history=pd.read_csv(ROOT/'training/results.csv');history.columns=history.columns.str.strip()
assert len(history)==80
display(pd.DataFrame([{'Setting':k,'Recorded value':str(args.get(k))} for k in ['model','epochs','batch','imgsz','optimizer','lr0','patience','workers','seed','fliplr','flipud','device']]))
print('Actual logged epochs:',len(history));display(history.tail())


## 7. Training and validation losses


In [ ]:
fig,axes=plt.subplots(2,3,figsize=(12,6))
for i,split in enumerate(['train','val']):
    for j,loss in enumerate(['box_loss','cls_loss','dfl_loss']):
        key=f'{split}/{loss}';axes[i,j].plot(history['epoch'],history[key]);axes[i,j].set_title(key);axes[i,j].set_xlabel('Epoch')
plt.tight_layout();plt.show()
figure('training/results.png')


## 8. Confidence-threshold evaluation
Observed validation F1: .25 → 99.3565%, .50 → 99.6129%, .70 → 99.7409%. Thus .70 has the highest F1 among these three tested settings. Recall decreases from100% to99.7409% (one additional missed sign; 0.2591 percentage points).

In [ ]:
thresholds=pd.read_csv(ROOT/'evaluation/validation_recomputed.csv')
display(thresholds[thresholds['class']=='ALL'])
figure('evaluation/figures/threshold_comparison.png')


## 9. ROC analysis and limitations
ROC uses a per-image, per-class score: maximum predicted confidence for that class at detector confidence floor .001 (zero when absent). Ground truth is class presence from validation annotations. Scores and labels are saved for independent recomputation. Image-presence ROC ignores localization; high AUC does not imply high detection mAP or transfer to external scenes.

In [ ]:
from roc_v2 import roc_curve,auc
saved=np.load(ROOT/'evaluation/validation_image_scores.npz')
assert saved['scores'].shape==saved['truth'].shape==(386,5)
roc=[]
for k,label in enumerate(CLASSES):
    fpr,tpr,_=roc_curve(saved['truth'][:,k],saved['scores'][:,k]);roc.append({'Class':label,'AUC':auc(fpr,tpr)})
display(pd.DataFrame(roc));figure('evaluation/figures/validation_roc.png')


## 10. Validation metrics

In [ ]:
def final_table(split):
    row=pd.read_csv(ROOT/f'evaluation/{split}_metrics.csv')
    display(row[['images','tp','fp','fn','precision_at_070','recall_at_070','f1_at_070','map50','map50_95']])
final_table('validation')
display(json.loads((ROOT/'evaluation/reproduction_check.json').read_text())['validation'])
display(pd.read_csv(ROOT/'evaluation/validation_recomputed.csv').query('threshold == .7'))


## 11. External testing metrics

In [ ]:
final_table('external')
display(json.loads((ROOT/'evaluation/reproduction_check.json').read_text())['external'])
display(pd.read_csv(ROOT/'evaluation/external_recomputed.csv'))


## 12. Confusion matrices and precision–recall curves

In [ ]:
for split in ['validation','external']:
    print(split,'— archived LOCAL model standard evaluation')
    figure(f'evaluation/figures/{split}/confusion_matrix.png',700)
    figure(f'evaluation/figures/{split}/PR_curve.png',700)
    figure(f'evaluation/figures/{split}_locked_confusion.png',700)


## 13. Sample detections
One unseen external example from each annotated class is shown below. Images are selected by ground-truth class, not by prediction success. Predictions use the submitted local checkpoint at .70. This is illustrative and does not replace aggregate evaluation.

In [ ]:
for k,label in enumerate(CLASSES):
    print(label);figure(f'evaluation/figures/sample_{k}.jpg',600)


## 14. Static-image inference
The prototype supports image upload, annotated output, a detected-sign table, and saving. The following safe cell predicts one included image and displays its suggested decisions; it opens no desktop window.

In [ ]:
example=next(iter(sorted((ROOT/'sample_images').glob('*'))))
result=model.predict(str(example),conf=CONFIDENCE,iou=NMS_IOU,imgsz=IMAGE_SIZE,device='cpu',verbose=False)[0]
import cv2
plt.figure(figsize=(7,5));plt.imshow(cv2.cvtColor(result.plot(),cv2.COLOR_BGR2RGB));plt.axis('off');plt.show()
print(decision_text([(CLASSES[int(b.cls.item())],float(b.conf.item())) for b in result.boxes]))


## 15. Live webcam implementation

Run locally: `python app.py`. The `app.py` opens cv2.VideoCapture(0) from an inference worker. A bounded queue sends frames/results to Tkinter's main thread via after callbacks. 

## 16. Rule-based Suggested Decision
Suggestions apply at confidence .70 and update on static and webcam results. Multiple distinct supported signs produce distinct suggestions. No detection produces No Decision. These are display-only messages; no robot controller or movement is implemented.

In [ ]:
display(pd.DataFrame(list(SIGN_DECISIONS.items()),columns=['Sign','Suggested Decision']))
expected=['STOP — Halt the robot','YIELD — Slow down and give way','DO NOT PROCEED — Route is prohibited','TURN LEFT — Prepare a left turn','TURN RIGHT — Prepare a right turn']
for label,value in zip(CLASSES,expected):assert get_suggested_decision(label,.7)==value
assert decision_text([])==NO_DECISION
assert get_suggested_decision('STOP',.69) is None
assert get_suggested_decision('unsupported',.99) is None
print('Decision mapping, threshold and no-detection checks passed.')
import app
assert app.MODEL_PATH==MODEL_PATH and app.CONFIDENCE==.7 and app.NMS_IOU==.7
print('GUI imports with final local checkpoint and settings; no window/camera opened.')


## 17. Discussion and limitations
Validation results are strong but external recall is only60.75%. Balanced training addresses count imbalance but reduces training diversity. Shared/near-identical scenes may inflate validation. Exact duplicates and annotation checks do not replace scene-level splitting or manual review. ROC measures image presence; detector mAP evaluates localization. Full-resolution annotation review, real webcam/FPS validation, source licensing review for redistribution, and reconciliation of the NSIT citation remain necessary. Threshold .70 was retained as requested, not proven universally optimal.

Archive provenance: best.pt SHA-256 matches the local row in the supplied validation comparison CSV. Archived evaluation folders identify Local_V2; mAP values were verified against these CSVs and not re-estimated in this packaging step. Fixed-threshold inference and ROC were rerun with the exact packaged checkpoint. The old model is not included as an alternate runtime fallback.